---


## 1. Environment Setup

To begin, we need to install the necessary libraries and set up the environment for the chatbot and backend system. The key libraries include:

- `transformers`: For pre-trained language models like BERT and GPT.
- `streamlit` or `flask`: For web-based GUI development.
- `nltk`, `spacy`, `tensorflow`: For NLP tasks like tokenization, intent classification, and text processing.

```bash
!pip install transformers streamlit nltk spacy tensorflow


# Student Services Chatbot Project Structure

## Directory Structure

```plaintext
student_services_chatbot/
├── data/
│   ├── raw_faqs.csv                # Original FAQs or student handbook data
│   ├── cleaned_faqs.csv            # Tokenized, normalized, labeled FAQ data
│   ├── test_queries.csv            # User input test queries for model evaluation
│
├── nlp_core/
│   ├── preprocessing.py            # Tokenization, stopword removal, lemmatization
│   ├── tfidf_engine.py             # Build TF, IDF, TF-IDF matrices
│   ├── language_model.py           # Unigram + Bigram model with Maximum Likelihood Estimation (MLE) and perplexity
│   ├── phrase_query.py             # Positional indexing and phrase search logic
│   ├── intent_classifier.py        # Naive Bayes or TF-IDF-based intent matching
│
├── semantic_layer/
│   ├── word2vec_engine.py          # CBOW/Skip-gram models for word embedding and similarity scoring
│   ├── embedding_utils.py          # Cosine similarity calculation and query vectorization
│
├── llm_fallback/
│   ├── cohere_handler.py           # Cohere API handler (Canadian-hosted LLM fallback)
│   ├── openai_handler.py           # OpenAI GPT-4 handler using LangChain
│   ├── fallback_router.py          # Confidence-based logic to trigger fallback if necessary
│
├── app/
│   ├── streamlit_app.py            # Web interface for user query input, top answer display, and fallback option
│   ├── query_logger.py             # Log user queries and feedback for further training
│
├── evaluation/
│   ├── metrics.py                  # Accuracy, perplexity, fallback rate, and match quality metrics
│   ├── visualizations.py           # Visualizations for vocabulary size, TF distribution, and heatmaps
│
├── models/
│   ├── tfidf_model.pkl             # Trained TF-IDF model
│   ├── bigram_model.pkl            # Trained Bigram model
│   ├── word2vec_model.model        # Pre-trained Word2Vec model
│
├── README.md                       # Full description of the project and instructions for setup
├── requirements.txt                # List of dependencies for the project (install via pip)
└── report/
    ├── final_poster.pdf            # Final project poster for presentation
    ├── code_snapshot.ipynb         # Executable summary notebook of the project
    └── presentation.pptx           # PowerPoint presentation summarizing the project


#### Loading spaCy's `en_core_web_sm` Model

This code loads a pre-trained English language model from **spaCy** to process and analyze English text, enabling tasks like tokenization, part-of-speech tagging, and named entity recognition.


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")


#### Step 1: Import Library
We import **PyMuPDF** (`fitz`) for extracting text from PDFs.

#### Step 2: Define Extraction Function
The `extract_text_from_pdf` function reads and extracts text from each page of the PDF.

#### Step 3: Extract Text from PDFs
Text is extracted from two PDF files: `RO_FAQ_Winter_2024.pdf` and `Student_Fees_FAQ_Winter_2024.pdf`.

#### Step 4: Preview Combined Text
The first 1000 characters of the combined text are displayed as a preview.


# Extracting Text from PDFs

## Step 1: Import PyMuPDF
Use **PyMuPDF** (fitz) to read PDF files.

## Step 2: Define Extraction Function
The `extract_text_from_pdf` function opens a PDF and extracts text from all pages.

## Step 3: Extract Text
Text from two PDFs (`RO_FAQ_Winter_2024.pdf` and `Student_Fees_FAQ_Winter_2024.pdf`) is extracted and combined.

## Step 4: Preview Text
The first 1000 characters of the combined text are printed for preview.

In [2]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

# Extract raw text from both files
faq_text_ro = extract_text_from_pdf("D:/proj/Data/RO_FAQ_Winter_2024.pdf")
faq_text_fees = extract_text_from_pdf("D:/proj/Data/Student_Fees_FAQ_Winter_2024.pdf")

# Combine and preview
combined_faq_text = faq_text_ro + "\n" + faq_text_fees
print(combined_faq_text[:1000])  # Preview first 1000 characters


 
  
Updated December 2023 
 
Office of the Registrar – Student Financial Services  
Contact Information  
OSAP Queries – osap@conestogac.on.ca  
Scholarships and Awards – awards@conestogac.on.ca 
Frequently Asked Questions  
 
Who is eligible for OSAP? 
For full-time students, you may be eligible for OSAP if you meet specific criteria. For more information, 
please visit the Student Financial Services website.  
Can I apply for OSAP as a part-time student? 
Yes, a student may be eligible for part-time OSAP if they meet specific criteria. For more information, 
please visit the Student Financial Services website. 
How do I apply for OSAP? 
You must apply for OSAP online. The application is usually released mid-Spring for the upcoming 
academic year. Learn how to apply for OSAP here.  
After I complete my application, what do I do next? 
Once your application is complete, check for required documents and do not delay in submitting them. 
Required documents can be uploaded directly to th

---

#### Extracting Q&A Pairs from Raw Text

The function `extract_qa_pairs` is designed to extract question-answer pairs from a raw text string, such as FAQ content.

#### Step 1: Split the Raw Text
The raw text is split into lines based on newline characters.

#### Step 2: Detect Questions and Collect Answers
- It detects lines ending with a `?` as questions.
- For each question, the function collects the answer, which may span multiple lines, until another question or blank line is encountered.

#### Step 3: Return DataFrame
The function returns a **Pandas DataFrame** with two columns: `question` and `answer`.

In [3]:
import re
import pandas as pd
def extract_qa_pairs(raw_text):
    lines = raw_text.split('\n')
    questions = []
    answers = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        # Detect question
        if line.endswith('?'):
            question = line
            answer = ""

            # Collect answer (might span multiple lines until next question or blank line)
            i += 1
            while i < len(lines):
                next_line = lines[i].strip()
                if next_line == "" or next_line.endswith('?'):
                    break
                answer += " " + next_line
                i += 1

            questions.append(question)
            answers.append(answer.strip())
        else:
            i += 1

    return pd.DataFrame({'question': questions, 'answer': answers})

# Run the extractor
faq_df = extract_qa_pairs(combined_faq_text)

# Preview extracted Q&A
faq_df.head(10)

def extract_qa_pairs(raw_text):
    lines = raw_text.split('\n')
    questions = []
    answers = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        # Detect question
        if line.endswith('?'):
            question = line
            answer = ""

            # Collect answer (might span multiple lines until next question or blank line)
            i += 1
            while i < len(lines):
                next_line = lines[i].strip()
                if next_line == "" or next_line.endswith('?'):
                    break
                answer += " " + next_line
                i += 1

            questions.append(question)
            answers.append(answer.strip())
        else:
            i += 1

    return pd.DataFrame({'question': questions, 'answer': answers})

# Run the extractor
faq_df = extract_qa_pairs(combined_faq_text)

# Preview extracted Q&A
faq_df.head(10)


,question,answer
0,Who is eligible for OSAP?,"For full-time students, you may be eligible fo..."
1,Can I apply for OSAP as a part-time student?,"Yes, a student may be eligible for part-time O..."
2,How do I apply for OSAP?,You must apply for OSAP online. The applicatio...
3,"After I complete my application, what do I do ...","Once your application is complete, check for r..."
4,How do I complete the Master Student Financial...,The Master Student Financial Assistance Agreem...
5,When is the OSAP application due?,The deadline to apply for OSAP is 60 days prio...
6,Do I need to reapply for OSAP each year?,You will need to reapply for OSAP each year yo...
7,How long does OSAP take to process and how do ...,The entire OSAP process can take 6 to 8 weeks ...
8,Do I need to pay the tuition deposit if I am a...,"Yes, all students are required to pay a non-re..."
9,Will my tuition fees be deferred if I am appro...,"Yes, the tuition due date will be deferred for..."


#### Text Preprocessing and Tokenization

This code cleans and tokenizes FAQ questions by:

1. **Cleaning**: Converts to lowercase, removes non-alphabetic characters, and trims extra spaces.
2. **Tokenization & Lemmatization**: Tokenizes text, removes stopwords, and lemmatizes tokens (using spaCy).


In [4]:
import re
import spacy
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_lemmatize(text):
    doc = nlp(text)
    return [
        token.lemma_ for token in doc
        if token.lemma_ not in ENGLISH_STOP_WORDS and len(token.lemma_) > 2
    ]

# Apply preprocessing
faq_df['cleaned'] = faq_df['question'].apply(clean_text)
faq_df['tokens'] = faq_df['cleaned'].apply(tokenize_lemmatize)

# Preview result
faq_df[['question', 'cleaned', 'tokens']].head()


,question,cleaned,tokens
0,Who is eligible for OSAP?,who is eligible for osap,"[eligible, osap]"
1,Can I apply for OSAP as a part-time student?,can i apply for osap as a parttime student,"[apply, osap, parttime, student]"
2,How do I apply for OSAP?,how do i apply for osap,"[apply, osap]"
3,"After I complete my application, what do I do ...",after i complete my application what do i do next,"[complete, application]"
4,How do I complete the Master Student Financial...,how do i complete the master student financial...,"[complete, master, student, financial, aid, ag..."


####  Save preprocessed Q&A with cleaned questions and tokens


In [5]:
faq_df.to_csv("./Data/faq_cleaned.csv", index=False)
print("✅ Cleaned FAQ saved to ./Data/faq_cleaned.csv")


✅ Cleaned FAQ saved to ./Data/faq_cleaned.csv


In [6]:
import pandas as pd

faq_df = pd.read_csv("./Data/faq_cleaned.csv")
faq_df.dropna(subset=['question', 'answer'], inplace=True)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use original cleaned questions
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(faq_df['cleaned'])

print("✅ TF-IDF matrix shape:", tfidf_matrix.shape)


✅ TF-IDF matrix shape: (27, 95)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def get_best_faq_match(user_query, top_k=3):
    user_clean = re.sub(r"[^a-zA-Z\s]", "", user_query.lower())
    user_tfidf = vectorizer.transform([user_clean])
    cosine_similarities = cosine_similarity(user_tfidf, tfidf_matrix).flatten()

    top_indices = cosine_similarities.argsort()[-top_k:][::-1]
    results = []
    for idx in top_indices:
        results.append({
            'question': faq_df.iloc[idx]['question'],
            'answer': faq_df.iloc[idx]['answer'],
            'score': cosine_similarities[idx]
        })
    return results


#### Query Matching and Displaying Results

This code matches a user query with the most relevant FAQ question and displays the result along with the similarity score.

1. **Query**: A sample user query is defined.
2. **Get Best Match**: The function `get_best_faq_match(query)` returns the most similar FAQ question-answer pair.
3. **Display Results**: For each match, the question, answer, and similarity score are printed.


In [9]:
query = "How do I drop my course?"
matches = get_best_faq_match(query)

for match in matches:
    print(f"🔹 Q: {match['question']}")
    print(f"   A: {match['answer']}")
    print(f"   🔢 Similarity Score: {match['score']:.4f}")
    print("---")


🔹 Q: How do I change my block or add/drop a course?
   A: If you are unable to make to changes yourself through your Portal, My Courses tab, then you will need to complete a digital Course Change Request form located in your Student Portal, Services tab and select the My Forms button.
   🔢 Similarity Score: 0.5195
---
🔹 Q: How do I pay for a course on Held Enrolment?
   A: Go to your Student Portal, select the Financial tab and then click on the Payment for Held Courses sub-tab. It is very important to make your payment within 72 hours or the hold will expire.
   🔢 Similarity Score: 0.4317
---
🔹 Q: How do I withdraw from my program?
   A: If you are withdrawing from your program, go to your Student Portal – Services tab and select the My Forms button. Then complete the digital Withdrawal Form. For withdrawal and refund information please visit: https://www.conestogac.on.ca/admissions/paying-your-fees/refunds-withdrawals Important Dates: • Last day to add/change course • Refund deadline

---

#### Chatbot Reply Logic

This function provides the chatbot's reply based on the user's query by matching it with the most relevant FAQ.

1. **Match FAQs**: The function uses `get_best_faq_match()` to find the top-k most similar FAQ questions.
2. **Confidence Check**: If the confidence score of the highest match is below a threshold, the bot returns a fallback message.
3. **Return Response**: Otherwise, it returns the answer, intent (matched question), and confidence score.


In [10]:
def chatbot_reply(user_query, top_k=1, min_confidence=0.2):
    matches = get_best_faq_match(user_query, top_k=top_k)
    
    # If highest match is too low, return fallback
    if matches[0]['score'] < min_confidence:
        return {
            "reply": "I'm not sure about that. Please speak with a Student Success Advisor.",
            "intent": "unknown",
            "confidence": matches[0]['score']
        }
    
    return {
        "reply": matches[0]['answer'],
        "intent": matches[0]['question'],
        "confidence": matches[0]['score']
    }

# Example usage
student_query = "Can I pay my tuition in installments?"
bot_response = chatbot_reply(student_query)

print(f"💬 User: {student_query}")
print(f"🤖 Bot: {bot_response['reply']}")
print(f"📌 Matched Q: {bot_response['intent']}")
print(f"📊 Confidence: {bot_response['confidence']:.4f}")


💬 User: Can I pay my tuition in installments?
🤖 Bot: Yes, all students are required to pay a non-refundable tuition deposit each academic year to secure a place in a program. For more information, visit Student Fee Invoices and Payment.
📌 Matched Q: Do I need to pay the tuition deposit if I am applying for OSAP?
📊 Confidence: 0.3788


In [11]:
# Example usage
student_query = "i play baseball ?  "
bot_response = chatbot_reply(student_query)

print(f"💬 User: {student_query}")
print(f"🤖 Bot: {bot_response['reply']}")
print(f"📌 Matched Q: {bot_response['intent']}")
print(f"📊 Confidence: {bot_response['confidence']:.4f}")

💬 User: i play baseball ?  
🤖 Bot: I'm not sure about that. Please speak with a Student Success Advisor.
📌 Matched Q: unknown
📊 Confidence: 0.0000


In [12]:
# Example usage
student_query = "can i ai ?  "
bot_response = chatbot_reply(student_query)

print(f"💬 User: {student_query}")
print(f"🤖 Bot: {bot_response['reply']}")
print(f"📌 Matched Q: {bot_response['intent']}")
print(f"📊 Confidence: {bot_response['confidence']:.4f}")

💬 User: can i ai ?  
🤖 Bot: While we are happy to discuss our processes and procedures with parents, guardians, agents (etc.), due to our privacy and confidentiality guidelines, we cannot disclose personal information, including academic and financial information, on behalf of a student, without their written, permission. If you would like to release your information to a designated individual, you can do so by completing a Consent Form, found on our website, and emailing the completed form to ClientServices@conestogac.on.ca
📌 Matched Q: Who can receive information about my account/ who can contact the College on my behalf?
📊 Confidence: 0.3341
